# Dual-stream architecture with asymmetric clinical safety evaluation — reproducibility notebook

Companion notebook for **Nasser-eddine et al.**, *"Dual-stream architecture with asymmetric clinical safety evaluation for colorectal histopathology"* (submitted to *Medical Image Analysis*).

**Self-contained**: every model and metric is defined inline; the only dependencies are `torch`, `numpy`, `scikit-learn`. It reproduces the paper's headline results for the three best configurations by ACSS (F4 selected, S4, F6) from precomputed embeddings, **without the raw images**:

| Level | F4 (selected) | S4 | F6 |
|---|---|---|---|
| **Patch** accuracy / ACSS / CAR→NOR | 92.37 % / 95.92 / 0.2 % | 93.39 % / 95.88 / 1.4 % | 92.67 % / 95.56 / 1.3 % |
| **Slide** (91 slides) accuracy / CAR→NOR | 92.8 ± 1.1 % / 0 of 205 | 95.0 ± 0.5 % / 1 of 205 | 92.3 ± 1.2 % / 0 of 205 |

F4 is selected not because it is the most accurate (S4 is) but because the ACSS ranks it first: it sends the fewest carcinomas to the *normal* class. Two contributions over the two-stage SupCon+focal baseline (Yengec-Tasdemir et al., 2024): the **dual-stream** encoder (each patch seen at native detail *and* as a `rebinned` 3×3 context view, fused through one weight-shared BiT-M R50x1) and the **ACSS**, plus a **Gated Attention MIL** aggregator at the slide level.

### Data and code availability

- **Code**: `https://github.com/oddissea/dualstream-crc`, tag `v1.0-media-submission` (archived on Zenodo, DOI `10.5281/zenodo.YYYYYYY`).
- **Embeddings, heads and slide caches** (one shared `test_embeddings.npz` — F4/S4/F6 use the same frozen encoder — plus per-model `<M>_head.pth` and `external_cache_<M>.pt`, M ∈ {F4, S4, F6}): Zenodo DOI `10.5281/zenodo.21327829`. Set `ZENODO_RECORD` below to auto-download the small artifacts.
- **Whole-slide images and patch-level image arrays**: available on reasonable request to the corresponding author, subject to HUC approval and a data-processing agreement — they contain patient material and are not publicly redistributable. See *Provenance* at the end for how the embeddings were produced.

## 0. Setup

The embeddings are the output of the **frozen** dual-stream encoder, so reproducing the paper needs only the small classifier head (patch level) and the attention aggregator (slide level) — both defined inline below. No encoder, no `timm`, no training framework.

In [1]:
import urllib.request
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold

device = torch.device('mps' if torch.backends.mps.is_available()
                      else 'cuda' if torch.cuda.is_available() else 'cpu')

# The three best configurations by ACSS. Per model: head, 91-slide cache, head dropout,
# and the paper's patch-level target (accuracy %, CAR->NOR %, ACSS).
# F4/S4/F6 share the SAME frozen SupCon encoder (identical stage-1 weights), so the test
# embeddings are identical -> one shared file; the models differ ONLY in the stage-2 head.
TEST_EMB = 'test_embeddings.npz'
MODELS = {
    'F4': dict(head='F4_head.pth', cache='external_cache_F4.pt', drop=0.1, paper=(92.37, 0.2, 95.92)),
    'S4': dict(head='S4_head.pth', cache='external_cache_S4.pt', drop=0.1, paper=(93.39, 1.4, 95.88)),
    'F6': dict(head='F6_head.pth', cache='external_cache_F6.pt', drop=0.2, paper=(92.67, 1.3, 95.56)),
}

# --- artifact resolution: local zenodo/ first, else download from Zenodo -------------
ZDIR = Path('zenodo')
ZENODO_RECORD = '21327829'   # <-- numeric Zenodo record id (fill in after depositing)

def resolve(filename):
    dest = ZDIR / filename
    if dest.exists():
        return dest
    if ZENODO_RECORD == 'XXXXXXX':
        raise FileNotFoundError(f'{dest} missing and ZENODO_RECORD not set — '
                                f'download from Zenodo or set the record id.')
    dest.parent.mkdir(parents=True, exist_ok=True)
    url = f'https://zenodo.org/records/{ZENODO_RECORD}/files/{filename}?download=1'
    print(f'downloading {filename} from Zenodo ...')
    def _progress(nblocks, bs, total):
        done = nblocks * bs
        pct = f' ({min(100, done * 100 // total)}%)' if total > 0 else ''
        print(f'\r  {done/1e6:,.0f} MB{pct}', end='', flush=True)
    tmp = dest.with_name(dest.name + '.part')   # atomic: rename in only when complete
    urllib.request.urlretrieve(url, tmp, reporthook=_progress)
    tmp.replace(dest)
    print(f'\r  {dest.stat().st_size/1e6:,.0f} MB  -> {dest}')
    return dest

print('device:', device)

device: mps


## 1. The two models (defined inline)

**Patch head** — the stage-2 classifier trained on the frozen 4096-d dual-stream embedding: one hidden layer (512-d, post-ReLU) and a 3-class output. This is the only thing the deposited `<M>_head.pth` weights load into.

**Gated Attention MIL** — the slide aggregator (Ilse et al., 2018): gated attention over the patch embeddings of a slide, weighted sum, then a small classifier. Trained from scratch on the 91-slide cache (§4).

In [2]:
class Head(nn.Module):
    """Stage-2 classifier: 4096 -> hidden 512 (post-ReLU) -> 3. Loads <M>_head.pth."""
    def __init__(self, embedding_dim=4096, hidden_units=512, num_classes=3, dropout_rate=0.1):
        super().__init__()
        self.hidden = nn.Linear(embedding_dim, hidden_units)
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(dropout_rate)
        self.output = nn.Linear(hidden_units, num_classes)
    def forward(self, x):
        return self.output(self.dropout(self.activation(self.hidden(x))))


class AttentionMIL(nn.Module):
    """Gated Attention MIL (Ilse et al., 2018): bag of patch embeddings -> slide logits."""
    def __init__(self, embedding_dim, hidden_dim=256, num_classes=3, dropout=0.25, heads=1):
        super().__init__()
        self.attention_V = nn.Sequential(nn.Linear(embedding_dim, hidden_dim), nn.Tanh())
        self.attention_U = nn.Sequential(nn.Linear(embedding_dim, hidden_dim), nn.Sigmoid())
        self.attention_W = nn.Linear(hidden_dim, heads)
        self.classifier  = nn.Sequential(nn.Linear(embedding_dim * heads, 256), nn.ReLU(),
                                         nn.Dropout(dropout), nn.Linear(256, num_classes))
    def forward(self, x):                       # x: (N patches, D)
        A = self.attention_W(self.attention_V(x) * self.attention_U(x))   # (N, heads)
        A = torch.softmax(A.transpose(0, 1), dim=1)                        # (heads, N)
        z = torch.mm(A, x).view(1, -1)                                     # (1, D*heads)
        return self.classifier(z), A

print('Head           ->', sum(p.numel() for p in Head().parameters()) / 1e3, 'K parameters')
print('AttentionMIL   ->', sum(p.numel() for p in AttentionMIL(512).parameters()) / 1e3, 'K parameters')

Head           -> 2099.203 K parameters
AttentionMIL   -> 395.012 K parameters


## 2. The Asymmetric Clinical Safety Score (ACSS)

Accuracy treats every error equally; in histopathology they are not. The ACSS weights each confusion by its **clinical cost** (elicited from the HUC Pathology Department). The worst confusion is **CAR→NOR** — a carcinoma called normal, leaving the patient without oncological follow-up — with weight 14.

$$\mathrm{ACSS} = 100 \times \left(1 - \frac{\sum_{i\neq j} P_{ij}\,C_{ij}}{\sum_k n_k \cdot \max_{j\neq k} P_{kj}}\right)$$

In [3]:
# Asymmetric penalty matrix P (rows=true, cols=pred; order NOR, ADE, CAR).
P = np.array([[ 0, 3, 6],     # NOR -> {NOR, ADE, CAR}
              [ 5, 0, 4],      # ADE -> {NOR, ADE, CAR}
              [14, 6, 0]],     # CAR -> {NOR, ADE, CAR}   (CAR->NOR = 14, most severe)
             dtype=float)

def acss(cm):
    """ACSS (0-100) from a 3x3 confusion matrix with rows=true in NOR/ADE/CAR order."""
    cm = np.asarray(cm, float)
    penal = (P * cm).sum()
    worst = sum(cm[k].sum() * P[k][np.arange(3) != k].max() for k in range(3))
    return 100.0 * (1 - penal / worst)

print('ACSS(perfect)       :', round(acss(np.diag([13127, 8819, 11828])), 2))
print('ACSS with 6 CAR->NOR:', round(acss([[100,0,0],[0,100,0],[6,0,94]]), 2))
print('ACSS with 6 NOR->ADE:', round(acss([[94,6,0],[0,100,0],[0,0,100]]), 2), '(same 6 errors, much safer)')

ACSS(perfect)       : 100.0
ACSS with 6 CAR->NOR: 96.64
ACSS with 6 NOR->ADE: 99.28 (same 6 errors, much safer)


## 3. Patch level — reproduce F4 from the deposited head + embeddings

`test_embeddings.npz` holds the 4096-d embeddings of the 6894 held-out test patches (the frozen dual-stream encoder, shared by F4/S4/F6). Applying the deposited `F4_head.pth` reproduces **92.37 %** accuracy, **ACSS 95.92** and **CAR→NOR 5/2412 (0.2 %)**. Label order in the `.npz`: `0=ADE, 1=NOR, 2=CAR`.

In [4]:
PERM_NAC = [1, 0, 2]    # ADE,NOR,CAR (model order) -> NOR,ADE,CAR (ACSS P order)

def patch_eval(m):
    """Load model m's deposited head + test embeddings; return patch-level metrics."""
    c = MODELS[m]
    head = Head(dropout_rate=c['drop'])
    head.load_state_dict(torch.load(resolve(c['head']), map_location='cpu')['classifier_state_dict'])
    head = head.to(device).eval()
    d = np.load(resolve(TEST_EMB))                              # shared across F4/S4/F6 (same encoder)
    with torch.no_grad():
        pred = head(torch.tensor(d['embeddings'], dtype=torch.float32).to(device)).argmax(1).cpu().numpy()
    cm = confusion_matrix(d['labels'], pred, labels=[0, 1, 2])  # order ADE, NOR, CAR
    return dict(acc=100 * accuracy_score(d['labels'], pred),
                acss=acss(cm[PERM_NAC][:, PERM_NAC]),           # reorder to NOR,ADE,CAR for P
                car_nor=int(cm[2, 1]), n_car=int(cm[2].sum()), n=len(d['labels']))  # true CAR(2) -> pred NOR(1)

r = patch_eval('F4')
print(f"test patches : {r['n']}")
print(f"accuracy     : {r['acc']:5.2f}%      (paper 92.37 %)")
print(f"ACSS         : {r['acss']:5.2f}       (paper 95.92)")
print(f"CAR->NOR     : {r['car_nor']}/{r['n_car']} = {100*r['car_nor']/r['n_car']:.1f}%   (paper 5/2412, 0.2 %)")

test patches : 6894
accuracy     : 92.37%      (paper 92.37 %)
ACSS         : 95.92       (paper 95.92)
CAR->NOR     : 5/2412 = 0.2%   (paper 5/2412, 0.2 %)


## 4. Family sweep — the ACSS separates what accuracy does not

F4 is the selected model, yet it is **not** the most accurate. Reproducing the patch-level ranking (Table 4) for the three best by ACSS: **S4 is one accuracy point above F4 but sends seven times as many carcinomas to the normal class** — exactly the trade-off the ACSS is built to surface.

The three configurations **share the same frozen encoder** (identical stage-1 embeddings) and differ only in the stage-2 head, so the entire spread in the table below comes from the head alone.

In [5]:
print(f"{'model':6}{'accuracy':>10}{'CAR->NOR':>14}{'ACSS':>8}     paper(acc, CAR%, ACSS)")
print('-' * 76)
for m in MODELS:
    r = patch_eval(m); p = MODELS[m]['paper']
    print(f"{m:6}{r['acc']:9.2f}%{r['car_nor']:>8}/{r['n_car']}={100*r['car_nor']/r['n_car']:>4.1f}%{r['acss']:8.2f}"
          f"     ({p[0]}, {p[1]}%, {p[2]})")
print('\nAccuracy would pick S4 (93.39%); the ACSS picks F4 (95.92): S4 misclassifies 7x as many\n'
      'carcinomas as normal (33 vs 5 of 2412). Clinical safety over raw accuracy.')

model   accuracy      CAR->NOR    ACSS     paper(acc, CAR%, ACSS)
----------------------------------------------------------------------------


F4        92.37%       5/2412= 0.2%   95.92     (92.37, 0.2%, 95.92)


S4        93.39%      33/2412= 1.4%   95.88     (93.39, 1.4%, 95.88)


F6        92.67%      31/2412= 1.3%   95.58     (92.67, 1.3%, 95.56)

Accuracy would pick S4 (93.39%); the ACSS picks F4 (95.92): S4 misclassifies 7x as many
carcinomas as normal (33 vs 5 of 2412). Clinical safety over raw accuracy.


## 5. Slide level — Gated Attention MIL over the 91-slide cohort

The **frozen** patch embeddings of the additional 91-slide cohort are aggregated with the Gated Attention MIL. Following the paper, we use the **512-d post-ReLU** embedding and run 5-fold cross-validation repeated over five seeds. This reproduces **92.8 ± 1.1 %** overall accuracy with **zero CAR→NOR** across the 205 carcinoma predictions for F4.

The cache label order is `NOR=0, ADE=1, CAR=2` (so CAR→NOR is confusion-matrix entry `[2, 0]`). Loading the 8 GB cache and training 5×5 aggregators takes a few minutes; swap `SLIDE_MODEL` to `'S4'`/`'F6'` to reproduce the rest of Table 6 (S4 → 95.0 %, 1/205 ; F6 → 92.3 %, 0/205).

In [6]:
SLIDE_MODEL = 'F4'                       # 'F4' | 'S4' | 'F6'
SEEDS = [42, 123, 456, 789, 2026]
FEAT_DIM = 512

def load_cache(path, key='embeddings_512'):
    cache = torch.load(path, map_location='cpu')
    code = {'NOR': 0, 'ADE': 1, 'TUM': 2}                  # TUM == CAR in the cache labels
    return {stem: dict(X=d[key], y=code[d['label']])
            for stem, d in cache['slides'].items()
            if d.get('group') == 'clinical'}               # the paper's 91-slide clinical cohort (41 CAR)

def set_seed(s):
    import random; random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.backends.mps.is_available(): torch.mps.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def train_fold(train_items, val_items, epochs=100, lr=1e-4, patience=15):
    model = AttentionMIL(FEAT_DIM).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    crit = nn.CrossEntropyLoss()
    best, no_imp, preds, ys = 0.0, 0, None, None
    for _ in range(epochs):
        model.train()
        for it in train_items:
            y = torch.tensor([it['y']], device=device)
            opt.zero_grad(); logits, _ = model(it['X'].to(device))
            crit(logits, y).backward(); opt.step()
        model.eval(); preds, ys = [], []
        with torch.no_grad():
            for it in val_items:
                logits, _ = model(it['X'].to(device))
                preds.append(int(logits.argmax(1))); ys.append(it['y'])
        acc = accuracy_score(ys, preds)
        if acc > best: best, no_imp = acc, 0
        else: no_imp += 1
        if no_imp >= patience: break
    return preds, ys                                        # last-epoch val predictions (as in the paper code)

def cv_one_seed(slides, seed, n_folds=5):
    set_seed(seed)
    stems = list(slides); ys = np.array([slides[s]['y'] for s in stems])
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    allp, ally = [], []
    for tr, va in skf.split(stems, ys):
        p, y = train_fold([slides[stems[i]] for i in tr], [slides[stems[i]] for i in va])
        allp += p; ally += y
    return confusion_matrix(ally, allp, labels=[0, 1, 2])   # aggregated over folds (rows NOR,ADE,CAR)

slides = load_cache(resolve(MODELS[SLIDE_MODEL]['cache']))
cms = [cv_one_seed(slides, s) for s in SEEDS]
accs = [100 * np.diag(cm).sum() / cm.sum() for cm in cms]
cm_sum = np.sum(cms, axis=0)
print(f"[{SLIDE_MODEL}] 91 slides, {len(SEEDS)} seeds x 5-fold CV")
print(f"  overall accuracy : {np.mean(accs):.2f} +/- {np.std(accs):.2f} %")
print(f"  CAR->NOR         : {int(cm_sum[2,0])}/{int(cm_sum[2].sum())}   (true CAR -> pred NOR)")

[F4] 91 slides, 5 seeds x 5-fold CV
  overall accuracy : 92.75 +/- 1.12 %
  CAR->NOR         : 0/205   (true CAR -> pred NOR)


## Provenance and reproducibility notes

- **How the embeddings were produced.** Each patch is encoded by the **frozen** dual-stream BiT-M R50x1 (stage-1 SupCon, τ=0.05); the head is trained in stage 2 with focal loss (γ=2.0, α by inverse square-root frequency) on the frozen embeddings. `test_embeddings.npz` holds the 4096-d encoder outputs on the held-out **test** patches; `external_cache_<M>.pt` the frozen embeddings of the additional 91-slide cohort. **F4/S4/F6 share the same stage-1 encoder** (Phase F/S only refine the stage-2 head), so a single `test_embeddings.npz` serves all three. The full training / from-images pipeline needs the raw images (request-only) and is available on request.
- **Selected model F4**, best by ACSS among the 41 patch-level runs. **S4** and **F6** are the next two best; §4 (patch) and §5 (slide) reproduce them.
- **To run without the raw data**: set `ZENODO_RECORD` (or drop the files into `zenodo/`). Patch level (§3/§4) needs the shared `test_embeddings.npz` (~88 MB) + each `<M>_head.pth` (~8 MB); slide level (§5) needs `external_cache_<M>.pt` (8 GB/model).
- **Dependencies**: `torch`, `numpy`, `scikit-learn`. Everything else (models, ACSS, CV) is defined in this notebook.
- Full, licensed code at `github.com/oddissea/dualstream-crc` (`v1.0-media-submission`), archived on Zenodo.